# Basic Workflow: Processing Raw Text with `inif`

This notebook demonstrates the basic `inif` workflow for processing raw text inputs with a base LLM tokenizer.

**Scenario:** A researcher has several text passages and wants to:
1. Tokenize them using a model's tokenizer
2. Find common token sequences across passages
3. Tag tokens of interest (e.g. numbers, named entities)
4. Store mock interpretability data (e.g. logit lens) on tagged tokens
5. Save and reload the enriched document

For a workflow starting from an **Inspect AI** evaluation log, see `inspect_workflow.ipynb`.

## Step 1: Convert raw text to inif format

The `from_texts` converter tokenizes each text string into a `Sample`, finds common token sequences across samples via set-intersection, and replaces them with compact sequence references.

We use the GPT-2 tokenizer here. You can substitute any HuggingFace tokenizer.

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

/Users/gsarti/Documents/projects/ndif-ecosystem/inif/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
from inif.converters.text import from_texts

texts = [
    (
        "The Eiffel Tower in Paris was built in 1889."
        " It stands 330 metres tall and attracts"
        " 7 million visitors each year."
    ),
    (
        "The Eiffel Tower in Paris is one of the most"
        " visited monuments in the world."
        " About 25000 tonnes of iron were used."
    ),
    (
        "The Eiffel Tower in Paris was designed by"
        " Gustave Eiffel for the 1889 World's Fair."
        " It took 2 years to build."
    ),
]

doc = from_texts(
    texts,
    tokenizer=tokenizer,
    sample_ids=["passage_0", "passage_1", "passage_2"],
    min_sequence_length=3,
)

print(f"InifDocument: {len(doc.samples)} samples")
print(f"Common sequences found: {len(doc.sequences)}")
for seq in doc.sequences:
    print(f"  '{seq.id}': {seq.tokens}")
print(f"Model: {doc.metadata.model.name}")

InifDocument: 3 samples
Common sequences found: 1
  'sequence_0': ['The', ' E', 'iff', 'el', ' Tower', ' in', ' Paris']
Model: gpt2


## Step 2: Inspect the document structure

Each sample contains tokens (with possible sequence references), the original text, and empty slots for spans, scores, and metadata.

In [3]:
for sample in doc.samples:
    n_refs = sum(1 for t in sample.tokens if t.is_sequence_ref)
    n_flat = len(sample.tokens) - n_refs
    print(f"Sample '{sample.id}':")
    print(f"  Tokens: {len(sample.tokens)} ({n_flat} flat, {n_refs} refs)")
    print(f"  Text preview: {sample.texts[0][:60]}...")
    tok_strs = [
        f"[{t.sequence_id}]" if t.is_sequence_ref else t.token for t in sample.tokens
    ]
    print(f"  Token strings: {tok_strs}")
    print()

Sample 'passage_0':
  Tokens: 19 (18 flat, 1 refs)
  Text preview: The Eiffel Tower in Paris was built in 1889. It stands 330 m...
  Token strings: ['[sequence_0]', ' was', ' built', ' in', ' 1889', '.', ' It', ' stands', ' 330', ' metres', ' tall', ' and', ' attracts', ' 7', ' million', ' visitors', ' each', ' year', '.']

Sample 'passage_1':
  Tokens: 21 (20 flat, 1 refs)
  Text preview: The Eiffel Tower in Paris is one of the most visited monumen...
  Token strings: ['[sequence_0]', ' is', ' one', ' of', ' the', ' most', ' visited', ' monuments', ' in', ' the', ' world', '.', ' About', ' 25', '000', ' tonnes', ' of', ' iron', ' were', ' used', '.']

Sample 'passage_2':
  Tokens: 23 (22 flat, 1 refs)
  Text preview: The Eiffel Tower in Paris was designed by Gustave Eiffel for...
  Token strings: ['[sequence_0]', ' was', ' designed', ' by', ' Gust', 'ave', ' E', 'iff', 'el', ' for', ' the', ' 1889', ' World', "'s", ' Fair', '.', ' It', ' took', ' 2', ' years', ' to', ' build', '.']



## Step 3: Expand sequences and tag tokens

Before tagging, expand sequence references so every token has a string to match against. Then use regex tagging to mark tokens of interest.

Two tagging modes are available:
- `tag_by_regex_all` — matches against individual token strings (fast, but misses BPE-split words)
- `tag_by_text_regex_all` — matches against concatenated text and maps back to constituent tokens (handles subword splits)

In [4]:
from inif import (
    expand_sequences,
    select_by_tag,
    tag_by_regex_all,
    tag_by_text_regex_all,
)

doc_expanded = expand_sequences(doc)

# Tag tokens containing digits (per-token regex is fine here)
tag_by_regex_all(doc_expanded, r"\d+", "number")

# Tag named entities using text-level regex to handle BPE splits
# (e.g. GPT-2 splits "Eiffel" into [" E", "iff", "el"])
tag_by_text_regex_all(doc_expanded, r"(?i)eiffel|paris|gustave", "entity")

for sample in doc_expanded.samples:
    numbers = select_by_tag(sample, "number")
    entities = select_by_tag(sample, "entity")
    print(f"Sample '{sample.id}':")
    print(
        f"  Numbers: {[t.token for t in numbers.tokens]}"
        f" at positions {numbers.positions}"
    )
    print(
        f"  Entities: {[t.token for t in entities.tokens]}"
        f" at positions {entities.positions}"
    )
    print()

Sample 'passage_0':
  Numbers: [' 1889', ' 330', ' 7'] at positions [10, 14, 19]
  Entities: [' E', 'iff', 'el', ' Paris'] at positions [1, 2, 3, 6]

Sample 'passage_1':
  Numbers: [' 25', '000'] at positions [19, 20]
  Entities: [' E', 'iff', 'el', ' Paris'] at positions [1, 2, 3, 6]

Sample 'passage_2':
  Numbers: [' 1889', ' 2'] at positions [17, 24]
  Entities: [' E', 'iff', 'el', ' Paris', ' Gust', 'ave', ' E', 'iff', 'el'] at positions [1, 2, 3, 6, 10, 11, 12, 13, 14]



## Step 4: Create spans from tagged tokens

Spans are named groups of token positions stored on the sample. They are useful for marking regions of interest (e.g. an answer span, a reasoning chain).

In [5]:
from inif import create_span_from_tag, select_by_span

for sample in doc_expanded.samples:
    create_span_from_tag(sample, "number", "numeric_tokens")
    create_span_from_tag(sample, "entity", "entity_tokens")

# Verify span-based selection works
sample = doc_expanded.samples[0]
print(f"Spans on '{sample.id}':")
for span in sample.spans:
    sel = select_by_span(sample, span.name)
    print(
        f"  '{span.name}': positions={span.positions},"
        f" tokens={[t.token for t in sel.tokens]}"
    )

Spans on 'passage_0':
  'numeric_tokens': positions=[10, 14, 19], tokens=[' 1889', ' 330', ' 7']
  'entity_tokens': positions=[1, 2, 3, 6], tokens=[' E', 'iff', 'el', ' Paris']


## Step 5: Attach mock interpretability data

In a real workflow, you would run logit lens (or another interpretability method) on the tagged positions using **nnterp** + **nnsight**:

```python
from nnterp import StandardizedTransformer
from nnterp.interventions import logit_lens

model = StandardizedTransformer("gpt2")

for sample in doc_expanded.samples:
    numbers = select_by_tag(sample, "number")
    results = logit_lens(model, sample.texts[0], token_idx=numbers.positions)
    for token, result in zip(numbers.tokens, results):
        token.__dict__["logit_lens"] = result
        token.model_extra["logit_lens"] = result
```

Here we mock the logit lens output.

In [6]:
import random

random.seed(42)
NUM_LAYERS = 4

for sample in doc_expanded.samples:
    for token in select_by_tag(sample, "number").tokens:
        logit_lens = {}
        for layer in range(NUM_LAYERS):
            prob = random.uniform(0.1, 0.9)
            top_tok = token.token if prob > 0.5 else "other"
            logit_lens[f"layer_{layer}"] = {
                "top_k": [{"token": top_tok, "prob": round(prob, 3)}]
            }
        token.__dict__["logit_lens"] = logit_lens
        if token.model_extra is not None:
            token.model_extra["logit_lens"] = logit_lens

# Show example
example = select_by_tag(doc_expanded.samples[0], "number").tokens[0]
print(f"Token '{example.token}' — logit lens data:")
for layer, data in getattr(example, "logit_lens", {}).items():
    top = data["top_k"][0]
    print(f"  {layer}: '{top['token']}' (prob={top['prob']})")

Token ' 1889' — logit lens data:
  layer_0: ' 1889' (prob=0.612)
  layer_1: 'other' (prob=0.12)
  layer_2: 'other' (prob=0.32)
  layer_3: 'other' (prob=0.279)


## Step 6: Save, validate, and reload

The enriched document (with tags, spans, and logit lens data) can be serialized to JSON. Extra fields on tokens survive the round-trip.

In [7]:
from inif import load, save, to_dict, validate

# Save to disk
save(doc_expanded, "eiffel_tower.inif.json")

# Validate against the JSON schema
validate(to_dict(doc_expanded))
print("Schema validation passed!")

# Reload and verify
doc_reloaded = load("eiffel_tower.inif.json")
print(f"Reloaded: {len(doc_reloaded.samples)} samples")

# Verify extra fields survived the round-trip
reloaded_nums = select_by_tag(doc_reloaded.samples[0], "number")
rt_token = reloaded_nums.tokens[0]
assert "logit_lens" in (rt_token.model_extra or {})
print(f"Logit lens data preserved for token '{rt_token.token}'")

Schema validation passed!
Reloaded: 3 samples
Logit lens data preserved for token ' 1889'


In [8]:
import json

# Preview the compact JSON (first sample, truncated tokens)
d = to_dict(doc_expanded, compact=True)
preview = {"metadata": d["metadata"], "samples": [d["samples"][0]]}
preview["samples"][0]["tokens"] = preview["samples"][0]["tokens"][:5]
print(json.dumps(preview, indent=2))

{
  "metadata": {
    "model": {
      "name": "gpt2"
    },
    "packages": {
      "inif": "0.1.dev0+d20260216",
      "transformers": "4.38.1"
    },
    "created_at": "2026-02-16T19:03:17.829646+00:00",
    "total_samples": 3
  },
  "samples": [
    {
      "id": "passage_0",
      "tokens": [
        {
          "id": 0,
          "token": "The",
          "sequence_id": "sequence_0"
        },
        {
          "id": 0,
          "token": " E",
          "sequence_id": "sequence_0",
          "tags": [
            "entity"
          ]
        },
        {
          "id": 0,
          "token": "iff",
          "sequence_id": "sequence_0",
          "tags": [
            "entity"
          ]
        },
        {
          "id": 0,
          "token": "el",
          "sequence_id": "sequence_0",
          "tags": [
            "entity"
          ]
        },
        {
          "id": 0,
          "token": " Tower",
          "sequence_id": "sequence_0"
        }
      ],
      "tex

In [9]:
# Cleanup
import os

os.remove("eiffel_tower.inif.json")